# Build Training Dataset

## Objective

Convert the normalized PostgreSQL tables into a model-ready analytical dataset.

The database contains:

- `calendar`
- `products`
- `stores`
- `prices`
- `sales`

The sales table contains approximately 58 million observations.

We will NOT load the complete dataset into pandas memory.

Instead:

1. PostgreSQL performs the joins.
2. We first test the query on a small sample.
3. The final dataset is retrieved in chunks.
4. Each chunk is saved as a Parquet file.

At this stage we are only preparing the analytical dataset.

Feature engineering, lag variables, rolling statistics and model training will be performed later.

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

warnings.filterwarnings("ignore")

# --------------------------------------------------
# Project root
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --------------------------------------------------
# Project configuration
# --------------------------------------------------

from src.database.config import DB_CONFIG

# --------------------------------------------------
# Display settings
# --------------------------------------------------

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

print("Project root:", PROJECT_ROOT)

Project root: d:\Mlprojects\Forecasting\Retail-Demand-Forecasting


In [2]:
from sqlalchemy.engine import URL

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=int(DB_CONFIG["port"]),
    database=DB_CONFIG["database"],
)

engine = sa.create_engine(
    db_url,
    pool_pre_ping=True,
)

with engine.connect() as conn:
    print("Connection successful!")
    print(
        "Database:",
        conn.execute(text("SELECT current_database();")).scalar()
    )
    print(
        "User:",
        conn.execute(text("SELECT current_user;")).scalar()
    )

Connection successful!
Database: retail_forecast_db
User: postgres


## 1. Check Database Scale

Before constructing the analytical dataset, we need to understand how large each table is.

This is important because the sales table contains tens of millions of rows.

We therefore avoid blindly loading the complete tables into pandas.

In [3]:
tables = [
    "calendar",
    "products",
    "stores",
    "prices",
    "sales",
]

row_counts = {}

with engine.connect() as conn:
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        row_counts[table] = count

for table, count in row_counts.items():
    print(f"{table:<12} {count:>15,}")

calendar               1,969
products               3,049
stores                    10
prices             6,841,121
sales             58,327,370


## 2. Validate Join Relationships

Our analytical dataset will be built from the following relationships:

sales → calendar
using `d`

sales → products
using `item_id`

sales → stores
using `store_id`

sales + calendar → prices
using:

- `item_id`
- `store_id`
- `wm_yr_wk`

Before performing the large join, we verify that these relationships do not unexpectedly multiply rows.

In [4]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT d) AS unique_days
FROM calendar;
"""

with engine.connect() as conn:
    result = conn.execute(text(query)).fetchone()

print("Calendar rows :", result.total_rows)
print("Unique d      :", result.unique_days)

Calendar rows : 1969
Unique d      : 1969


In [5]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (item_id, store_id, wm_yr_wk)) AS unique_keys
FROM prices;
"""

with engine.connect() as conn:
    result = conn.execute(text(query)).fetchone()

print("Price rows       :", f"{result.total_rows:,}")
print("Unique price keys:", f"{result.unique_keys:,}")

Price rows       : 6,841,121
Unique price keys: 6,841,121


## 3. Define the Analytical Query

Instead of loading five large tables into pandas and repeatedly using `merge()`, PostgreSQL will perform the joins.

The result will contain:

### Identifiers

- item_id
- store_id
- d
- date

### Target

- sales_quantity

### Product information

- dept_id
- cat_id

### Store information

- state_id

### Calendar information

- wm_yr_wk
- weekday
- wday
- month
- year
- events
- SNAP indicators

### Price

- sell_price

No forecasting features are created here.

In [6]:
ANALYTICAL_QUERY = """
SELECT
    s.item_id,
    s.store_id,
    s.d,

    s.sales_quantity,

    c.date,
    c.wm_yr_wk,
    c.weekday,
    c.wday,
    c.month,
    c.year,

    c.event_name_1,
    c.event_type_1,
    c.event_name_2,
    c.event_type_2,

    c."snap_CA",
    c."snap_TX",
    c."snap_WI",

    p.sell_price,

    pr.dept_id,
    pr.cat_id,

    st.state_id

FROM sales AS s

LEFT JOIN calendar AS c
    ON s.d = c.d

LEFT JOIN prices AS p
    ON s.item_id = p.item_id
    AND s.store_id = p.store_id
    AND c.wm_yr_wk = p.wm_yr_wk

LEFT JOIN products AS pr
    ON s.item_id = pr.item_id

LEFT JOIN stores AS st
    ON s.store_id = st.store_id
"""

## 4. Test the Analytical Query

We do not immediately process all 58 million rows.

First we execute the query on a small sample.

If the SQL query, joins and column names are correct, we can safely move to the full chunked extraction.

In [7]:
TEST_QUERY = ANALYTICAL_QUERY + """
LIMIT 100000
"""

test_df = pd.read_sql_query(
    TEST_QUERY,
    engine,
)

print("Shape:", test_df.shape)

display(test_df.head())

Shape: (100000, 21)


,item_id,store_id,d,sales_quantity,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,dept_id,cat_id,state_id
0,FOODS_3_208,CA_3,d_8,0,2011-02-05,11102,Saturday,1,2,2011,NaN,NaN,NaN,NaN,True,True,True,NaN,FOODS_3,FOODS,CA
1,FOODS_2_104,TX_2,d_10,4,2011-02-07,11102,Monday,3,2,2011,NaN,NaN,NaN,NaN,True,True,False,6.98,FOODS_2,FOODS,TX
2,FOODS_2_272,TX_2,d_10,0,2011-02-07,11102,Monday,3,2,2011,NaN,NaN,NaN,NaN,True,True,False,NaN,FOODS_2,FOODS,TX
3,FOODS_3_015,TX_2,d_10,0,2011-02-07,11102,Monday,3,2,2011,NaN,NaN,NaN,NaN,True,True,False,NaN,FOODS_3,FOODS,TX
4,FOODS_3_043,TX_2,d_10,0,2011-02-07,11102,Monday,3,2,2011,NaN,NaN,NaN,NaN,True,True,False,NaN,FOODS_3,FOODS,TX


In [8]:
print("Columns:")
for column in test_df.columns:
    print("-", column)

print("\nData types:")
display(test_df.dtypes)

Columns:
- item_id
- store_id
- d
- sales_quantity
- date
- wm_yr_wk
- weekday
- wday
- month
- year
- event_name_1
- event_type_1
- event_name_2
- event_type_2
- snap_CA
- snap_TX
- snap_WI
- sell_price
- dept_id
- cat_id
- state_id

Data types:


item_id               str
store_id              str
d                     str
sales_quantity      int64
date               object
wm_yr_wk            int64
weekday               str
wday                int64
month               int64
year                int64
event_name_1          str
event_type_1          str
event_name_2          str
event_type_2          str
snap_CA              bool
snap_TX              bool
snap_WI              bool
sell_price        float64
dept_id               str
cat_id                str
state_id              str
dtype: object

In [9]:
missing = (
    test_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing[missing > 0])

event_name_2    99523
event_type_2    99523
event_type_1    92008
event_name_1    92008
sell_price      50384
dtype: int64

In [10]:
print("Target statistics:")

display(
    test_df["sales_quantity"].describe()
)

print(
    "\nZero-sales percentage:",
    round(
        (test_df["sales_quantity"] == 0).mean() * 100,
        2
    ),
    "%"
)

Target statistics:


count    100000.000000
mean          0.854950
std           3.396637
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         162.000000
Name: sales_quantity, dtype: float64


Zero-sales percentage: 77.75 %


In [11]:
print("Test rows:", len(test_df))

print(
    "Unique sales keys:",
    test_df[
        ["item_id", "store_id", "d"]
    ].drop_duplicates().shape[0]
)

Test rows: 100000
Unique sales keys: 100000


## 5. Full Dataset Extraction

The test query worked, so we can now process the full dataset.

We still will NOT create a 58-million-row pandas DataFrame.

Instead, pandas will retrieve the SQL result in chunks.

Each chunk will be:

1. validated
2. lightly processed
3. written to a Parquet file
4. released from memory

This keeps memory usage bounded.

In [12]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "training_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Output directory:")
print(OUTPUT_DIR)

Output directory:
d:\Mlprojects\Forecasting\Retail-Demand-Forecasting\data\processed\training_dataset


In [ ]:
CHUNK_SIZE = 50_000

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "training_dataset"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Remove old parquet parts
for file in OUTPUT_DIR.glob("part_*.parquet"):
    file.unlink()

total_rows = 0
part_number = 0

for chunk_df in pd.read_sql_query(
    ANALYTICAL_QUERY,
    engine,
    chunksize=CHUNK_SIZE,
):

    if chunk_df.empty:
        continue

    output_file = (
        OUTPUT_DIR / f"part_{part_number:05d}.parquet"
    )

    chunk_df.to_parquet(
        output_file,
        index=False,
        engine="pyarrow",
    )

    total_rows += len(chunk_df)

    print(
        f"Part {part_number:05d} | "
        f"Rows: {len(chunk_df):,} | "
        f"Total: {total_rows:,}"
    )

    del chunk_df

print("\nExtraction completed.")
print("Total rows:", f"{total_rows:,}")
print("Parts:", part_number)

In [ ]:
## 6. Verify the Generated Dataset

The output is stored as a Parquet dataset consisting of multiple files.

We deliberately keep the files separate rather than combining all 58 million rows into one pandas DataFrame.

This allows later ML processing to read only the required columns or subsets of the data.

In [ ]:
import pyarrow.dataset as ds

parquet_dataset = ds.dataset(
    OUTPUT_DIR,
    format="parquet",
)

fragments = list(
    parquet_dataset.get_fragments()
)

print("Number of parquet parts:", len(fragments))
print("\nSchema:")
print(parquet_dataset.schema)